In [ ]:


import numpy as np 
import pandas as pd 


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub


In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os

print(os.listdir('/kaggle/input'))

In [ ]:
import os

print(os.listdir('/kaggle/input/datasets'))

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split

dataset_path = "/kaggle/input/datasets/awsaf49/mnist-dataset/mnist_png/training"

images = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    labels=None,
    color_mode="grayscale",
    image_size=(28, 28),
    batch_size=None,
    shuffle=True
)

images = np.array(list(images))

images = images.astype("float32") / 255.0

x_train, x_test = train_test_split(
    images,
    test_size=0.2,
    random_state=42
)

print("Training:", x_train.shape)
print("Testing :", x_test.shape)

In [ ]:
noise_factor = 0.5

x_train_noisy = x_train + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=x_train.shape
)

x_test_noisy = x_test + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=x_test.shape
)

x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_test_noisy = np.clip(x_test_noisy, 0., 1.)

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Model

input_img = Input(shape=(28,28,1))

# Encoder
x = Conv2D(32,3,activation='relu',padding='same')(input_img)
x = MaxPooling2D(2,padding='same')(x)

x = Conv2D(16,3,activation='relu',padding='same')(x)
encoded = MaxPooling2D(2,padding='same')(x)

# Decoder
x = Conv2D(16,3,activation='relu',padding='same')(encoded)
x = UpSampling2D(2)(x)

x = Conv2D(32,3,activation='relu',padding='same')(x)
x = UpSampling2D(2)(x)

decoded = Conv2D(1,3,activation='sigmoid',padding='same')(x)

autoencoder = Model(input_img, decoded)

autoencoder.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

autoencoder.summary()

In [ ]:
history = autoencoder.fit(
    x_train_noisy,
    x_train,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test)
)

In [ ]:
decoded_imgs = autoencoder.predict(x_test_noisy)

In [ ]:
import matplotlib.pyplot as plt

n = 10
plt.figure(figsize=(20,6))

for i in range(n):

    # Noisy
    ax = plt.subplot(3,n,i+1)
    plt.imshow(x_test_noisy[i].reshape(28,28), cmap='gray')
    plt.axis("off")

    # Original
    ax = plt.subplot(3,n,i+1+n)
    plt.imshow(x_test[i].reshape(28,28), cmap='gray')
    plt.axis("off")

    # Denoised
    ax = plt.subplot(3,n,i+1+2*n)
    plt.imshow(decoded_imgs[i].reshape(28,28), cmap='gray')
    plt.axis("off")

plt.show()

In [ ]:
loss = autoencoder.evaluate(x_test_noisy, x_test)

print("Test Loss:", loss)

In [ ]:
autoencoder.save("mnist_denoising_autoencoder.keras")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)

plt.show()